In [ ]:
#Install + Imports (run first)

!pip install -U albumentations==2.0.8 opencv-python-headless
!pip install -q ultralytics timm

import os, json, glob, shutil, warnings, time
from pathlib import Path
import numpy as np
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder

import timm
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

from ultralytics import YOLO
warnings.filterwarnings('ignore')

In [ ]:
#Config+Helpers

# Paths
KAGGLE_INPUT = "/kaggle/input"
KAGGLE_WORKING = "/kaggle/working"

ROOT_DIR   = f"{KAGGLE_INPUT}/final-thesis-data/P3_Final_Data"
IMAGES_DIR = os.path.join(ROOT_DIR, "images")
LABELS_DIR = os.path.join(ROOT_DIR, "labels")

OUTPUT_BASE = f"{KAGGLE_WORKING}/PipelineData"
STAGE1_DIR  = os.path.join(OUTPUT_BASE, "Stage1_YOLOv8_Binary")
STAGE2_DIR  = os.path.join(OUTPUT_BASE, "Stage2_EfficientNet_Multiclass")
MODELS_DIR  = os.path.join(OUTPUT_BASE, "Models")
RESULTS_DIR = os.path.join(OUTPUT_BASE, "Results")

for p in [STAGE1_DIR, STAGE2_DIR, MODELS_DIR, RESULTS_DIR]:
    os.makedirs(p, exist_ok=True)

# 4 classes
CLASS_ID_TO_NAME = {
    0: 'Hole',
    1: 'Knot-slub',
    2: 'Thick/Missing yarn',
    3: 'Spot'
}

def is_image_file(fname): return fname.lower().endswith(('.jpg', '.jpeg', '.png'))
def is_empty_label(label_path): return (not os.path.exists(label_path)) or os.stat(label_path).st_size == 0

def convert_to_binary_labels(input_label_path, output_label_path):
    # YOLO binary: class 0 = any defect; empty file = no defect
    if is_empty_label(input_label_path):
        open(output_label_path, 'w').close()
        return
    with open(input_label_path, 'r') as f_in, open(output_label_path, 'w') as f_out:
        for line in f_in:
            parts = line.strip().split()
            if len(parts) == 5:
                f_out.write(f"0 {' '.join(parts[1:])}\n")

def crop_by_label(image_path, label_path, output_dir, base_name, padding=20):
    """Crop GT boxes (stage2 train/val/test creation)."""
    if is_empty_label(label_path): return 0
    img = cv2.imread(image_path); 
    if img is None: return 0
    h, w = img.shape[:2]; count = 0

    with open(label_path, 'r') as f:
        for i, line in enumerate(f):
            parts = line.strip().split()
            if len(parts) != 5: continue
            class_id, xc, yc, ww, hh = map(float, parts); class_id = int(class_id)
            if class_id not in (0,1,2,3): continue

            xc, yc, ww, hh = xc*w, yc*h, ww*w, hh*h
            x1 = max(0, int(xc - ww/2) - padding)
            y1 = max(0, int(yc - hh/2) - padding)
            x2 = min(w, int(xc + ww/2) + padding)
            y2 = min(h, int(yc + hh/2) + padding)
            crop = img[y1:y2, x1:x2]
            if crop.size == 0: continue

            class_folder = os.path.join(output_dir, f"class_{class_id}")
            os.makedirs(class_folder, exist_ok=True)
            out = os.path.join(class_folder, f"{base_name}_defect_{i}.jpg")
            cv2.imwrite(out, crop); count += 1
    return count


In [ ]:
#Dataset Audit

image_files = sorted([f for f in os.listdir(IMAGES_DIR) if is_image_file(f)])
label_files = sorted([f for f in os.listdir(LABELS_DIR) if f.endswith('.txt')])

img_set = set(Path(x).stem for x in image_files)
lbl_set = set(Path(x).stem for x in label_files)

print("📊 DATASET VALIDATION")
print(f"Images: {len(image_files)} | Labels: {len(label_files)} | Pairs: {len(img_set & lbl_set)}")
print(f"Orphan images: {len(img_set - lbl_set)} | Orphan labels: {len(lbl_set - img_set)}")


In [ ]:
#Build 70/15/15 splits + Stage-1/Stage-2 datasets

# 70/15/15 split
all_imgs = [f for f in os.listdir(IMAGES_DIR) if is_image_file(f)]
train_imgs, temp_imgs = train_test_split(all_imgs, test_size=0.30, random_state=42)   # 70% / 30%
val_imgs, test_imgs   = train_test_split(temp_imgs, test_size=0.50, random_state=42) # 15% / 15%

splits = {'train': train_imgs, 'val': val_imgs, 'test': test_imgs}

# Create folder structures
for split in splits:
    os.makedirs(os.path.join(STAGE1_DIR, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(STAGE1_DIR, split, "labels"), exist_ok=True)
    for cid in range(4):
        os.makedirs(os.path.join(STAGE2_DIR, split, f"class_{cid}"), exist_ok=True)

# Populate datasets
for split, files in splits.items():
    for fname in files:
        base = Path(fname).stem
        img_src = os.path.join(IMAGES_DIR, fname)
        lbl_src = os.path.join(LABELS_DIR, f"{base}.txt")

        # Stage-1 (binary YOLO)
        img_dst = os.path.join(STAGE1_DIR, split, "images", fname)
        lbl_dst = os.path.join(STAGE1_DIR, split, "labels", f"{base}.txt")
        shutil.copy(img_src, img_dst)
        convert_to_binary_labels(lbl_src, lbl_dst)

        # Stage-2 (GT crops → multi-class classifier)
        crop_dir = os.path.join(STAGE2_DIR, split)
        crop_by_label(img_src, lbl_src, crop_dir, base)

# Write YOLO data.yaml
with open(os.path.join(STAGE1_DIR, "data.yaml"), "w") as f:
    f.write(f"path: {STAGE1_DIR}\ntrain: train/images\nval: val/images\ntest: test/images\nnc: 1\nnames: ['defective']\n")

print("✅ Stage-1 (binary) and Stage-2 (multiclass) datasets built with 70/15/15.")


In [ ]:
#Stage-1 (YOLOv8) train + validate (robust weights path)

def train_yolo_stage1(model_name='yolov8n.pt', epochs=20, batch=16):
    print("🚀 Training YOLOv8 (binary)")
    model = YOLO(model_name)
    model.train(
        data=os.path.join(STAGE1_DIR, 'data.yaml'),
        epochs=epochs, imgsz=640, batch=batch,
        device=0 if torch.cuda.is_available() else 'cpu',
        project=MODELS_DIR, name='yolo_stage1', save=True, plots=False,
        patience=10, save_period=1
    )
    # Resolve best.pt robustly
    save_dir = getattr(getattr(model, 'trainer', None), 'save_dir', None)
    if save_dir and os.path.exists(os.path.join(save_dir, 'weights', 'best.pt')):
        best = os.path.join(save_dir, 'weights', 'best.pt')
    else:
        cands = glob.glob(os.path.join(MODELS_DIR, "yolo_stage1*", "weights", "best.pt"))
        if not cands: raise FileNotFoundError("best.pt not found under MODELS_DIR")
        best = sorted(cands, key=os.path.getmtime)[-1]

    target = os.path.join(MODELS_DIR, 'yolo_stage1_best.pt')
    shutil.copy(best, target)
    print(f"✅ Best weights: {target}")
    return YOLO(target)

def validate_yolo_stage1():
    print("🔍 Validating YOLOv8 on test split")
    model_path = os.path.join(MODELS_DIR, 'yolo_stage1_best.pt')
    model = YOLO(model_path)
    res = model.val(
        data=os.path.join(STAGE1_DIR, 'data.yaml'),
        split='test', project=RESULTS_DIR, name='yolo_stage1_val', save=True, plots=True
    )
    precision = float(getattr(res.box, "mp", 0.0))
    recall    = float(getattr(res.box, "mr", 0.0))
    mAP50     = float(getattr(res.box, "map50", 0.0))
    mAP50_95  = float(getattr(res.box, "map", 0.0))
    return dict(precision=precision, recall=recall, mAP50=mAP50, mAP50_95=mAP50_95)

yolo_model = train_yolo_stage1(model_name='yolov8n.pt', epochs=20, batch=16)
stage1_metrics = validate_yolo_stage1()
print(stage1_metrics)


In [ ]:
#Cropping for Stage-2 inference (keep out of evaluation)

def crop_defected_regions_for_inference():
    """Run YOLO on Stage-1 test images and save crops for manual inspection/inference only."""
    print("✂️  Cropping YOLO detections for inference only")
    yolo = YOLO(os.path.join(MODELS_DIR, 'yolo_stage1_best.pt'))
    test_images_dir = os.path.join(STAGE1_DIR, 'test', 'images')  # ✅ fixed
    if not os.path.exists(test_images_dir):
        raise FileNotFoundError(test_images_dir)

    infer_dir = os.path.join(STAGE2_DIR, 'inference', 'unknown')
    os.makedirs(infer_dir, exist_ok=True)

    results = yolo.predict(test_images_dir, save=False)
    meta = []; n = 0
    for r in results:
        img_path = r.path
        img = cv2.imread(img_path)
        if img is None: continue
        stem = Path(img_path).stem
        if r.boxes is not None and len(r.boxes) > 0:
            for j, box in enumerate(r.boxes):
                x1,y1,x2,y2 = box.xyxy[0].cpu().numpy().astype(int)
                crop = img[y1:y2, x1:x2]
                if crop.size == 0: continue
                out = os.path.join(infer_dir, f"{stem}_crop{j}.jpg")
                cv2.imwrite(out, crop); n += 1
                meta.append({"src": img_path, "crop": out, "bbox":[int(x1),int(y1),int(x2),int(y2)]})
    with open(os.path.join(infer_dir,'crop_metadata.json'), 'w') as f:
        json.dump(meta, f, indent=2)
    print(f"✅ {n} crops written to {infer_dir}")

crop_defected_regions_for_inference()


In [ ]:
#Stage-2 (EfficientNet-B0) train/validate/test + PR/F1 + plots

class EfficientNetStage2:
    def __init__(self, model_name='efficientnet_b0', num_classes=4):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = timm.create_model(model_name, pretrained=True, num_classes=num_classes).to(self.device)

        self.train_tf = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.5),
            transforms.RandomRotation(10),
            transforms.ColorJitter(0.1,0.1,0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
        self.val_tf = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])

        self.crit = nn.CrossEntropyLoss()
        self.opt  = optim.AdamW(self.model.parameters(), lr=1e-3, weight_decay=1e-2)
        self.sched = optim.lr_scheduler.ReduceLROnPlateau(self.opt, mode='min', factor=0.5, patience=3, verbose=True)
        self.class_names = None

    def load_data(self):
        train_ds = ImageFolder(os.path.join(STAGE2_DIR, 'train'), transform=self.train_tf)
        val_ds   = ImageFolder(os.path.join(STAGE2_DIR, 'val'),   transform=self.val_tf)
        test_ds  = ImageFolder(os.path.join(STAGE2_DIR, 'test'),  transform=self.val_tf)

        self.train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2)
        self.val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2)
        self.test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False, num_workers=2)

        self.class_names = train_ds.classes  # ['class_0', ...]
        print("Classes:", self.class_names)
        return self.class_names

    def train_epoch(self):
        self.model.train(); tl=0; corr=0; tot=0
        for x,y in tqdm(self.train_loader, desc="Train", leave=False):
            x,y = x.to(self.device), y.to(self.device)
            self.opt.zero_grad()
            out = self.model(x)
            loss = self.crit(out,y); loss.backward(); self.opt.step()
            tl += loss.item()
            corr += (out.argmax(1)==y).sum().item()
            tot += y.size(0)
        return tl/len(self.train_loader), 100*corr/tot

    def validate(self):
        self.model.eval(); tl=0; corr=0; tot=0; preds=[]; targs=[]
        with torch.no_grad():
            for x,y in tqdm(self.val_loader, desc="Val", leave=False):
                x,y = x.to(self.device), y.to(self.device)
                out = self.model(x)
                loss = self.crit(out,y); tl += loss.item()
                p = out.argmax(1)
                corr += (p==y).sum().item(); tot += y.size(0)
                preds.extend(p.cpu().tolist()); targs.extend(y.cpu().tolist())
        acc = 100*corr/tot
        pre = precision_score(targs, preds, average='weighted', zero_division=0)
        rec = recall_score(targs, preds, average='weighted', zero_division=0)
        f1  = f1_score(targs, preds, average='weighted', zero_division=0)
        return tl/len(self.val_loader), acc, pre, rec, f1

    def fit(self, epochs=20, patience=5):
        best = -1; patience_ctr=0
        trL, vaL, trA, vaA = [], [], [], []
        for ep in range(epochs):
            print(f"\nEpoch {ep+1}/{epochs}")
            tl, ta = self.train_epoch()
            vl, va, vp, vr, vf1 = self.validate()
            self.sched.step(vl)
            trL.append(tl); vaL.append(vl); trA.append(ta); vaA.append(va)
            print(f"Train loss {tl:.4f} acc {ta:.2f}% | Val loss {vl:.4f} acc {va:.2f}% P {vp:.3f} R {vr:.3f} F1 {vf1:.3f}")
            if va > best:
                best = va; patience_ctr=0
                torch.save(self.model.state_dict(), os.path.join(MODELS_DIR,'efficientnet_stage2_best.pth'))
                print("✅ Saved new best Stage-2 model")
            else:
                patience_ctr += 1
                if patience_ctr >= patience:
                    print("⏹️ Early stopping"); break
        # curves
        plt.figure(figsize=(12,4))
        plt.subplot(1,2,1); plt.plot(trL,label='train'); plt.plot(vaL,label='val'); plt.title('Loss'); plt.legend()
        plt.subplot(1,2,2); plt.plot(trA,label='train'); plt.plot(vaA,label='val'); plt.title('Accuracy'); plt.legend()
        plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR,'stage2_training_curves.png'), dpi=150)
        return best

    def test(self):
        self.model.load_state_dict(torch.load(os.path.join(MODELS_DIR,'efficientnet_stage2_best.pth'), map_location=self.device))
        self.model.eval(); preds=[]; targs=[]
        with torch.no_grad():
            for x,y in tqdm(self.test_loader, desc="Test"):
                x,y = x.to(self.device), y.to(self.device)
                out = self.model(x)
                preds.extend(out.argmax(1).cpu().tolist())
                targs.extend(y.cpu().tolist())
        # Metrics
        acc = accuracy_score(targs, preds)
        pre = precision_score(targs, preds, average='weighted', zero_division=0)
        rec = recall_score(targs, preds, average='weighted', zero_division=0)
        f1  = f1_score(targs, preds, average='weighted', zero_division=0)
        print(f"\n📊 Stage-2 Test — Acc {acc:.3f}  P {pre:.3f}  R {rec:.3f}  F1 {f1:.3f}")

        # Classification report with readable names
        # Map ImageFolder class_0..3 to human names:
        readable = [CLASS_ID_TO_NAME[int(c.split('_')[1])] for c in self.class_names]
        print(classification_report(targs, preds, target_names=readable, zero_division=0))

        # Confusion matrix
        cm = confusion_matrix(targs, preds)
        plt.figure(figsize=(8,6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=readable, yticklabels=readable)
        plt.title('Confusion Matrix — Stage-2')
        plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout()
        plt.savefig(os.path.join(RESULTS_DIR,'stage2_confusion_matrix.png'), dpi=150)

        return dict(accuracy=acc, precision=pre, recall=rec, f1=f1)

trainer = EfficientNetStage2()
trainer.load_data()
_ = trainer.fit(epochs=20, patience=5)
stage2_metrics = trainer.test()


In [ ]:
#Final JSON Results Mapping

pipeline_results = {
    "stage1_yolo": stage1_metrics,
    "stage2_efficientnet": stage2_metrics,
    "pipeline_mode": "train_and_infer",
    "gpu_used": torch.cuda.is_available()
}
with open(os.path.join(RESULTS_DIR, 'pipeline_results.json'), 'w') as f:
    json.dump(pipeline_results, f, indent=2)
print("✅ Results saved:", os.path.join(RESULTS_DIR, 'pipeline_results.json'))


In [ ]:
#Unit Tests

def assert_exists(path): 
    assert os.path.exists(path), f"Missing path: {path}"

def test_structure():
    # Stage-1
    for split in ['train','val','test']:
        assert_exists(os.path.join(STAGE1_DIR, split, 'images'))
        assert_exists(os.path.join(STAGE1_DIR, split, 'labels'))
    assert_exists(os.path.join(STAGE1_DIR, 'data.yaml'))

    # Stage-2
    for split in ['train','val','test']:
        for c in range(4):
            assert_exists(os.path.join(STAGE2_DIR, split, f'class_{c}'))
    print("✅ test_structure passed")

def test_models_present():
    assert_exists(os.path.join(MODELS_DIR,'yolo_stage1_best.pt'))
    assert_exists(os.path.join(MODELS_DIR,'efficientnet_stage2_best.pth'))
    print("✅ test_models_present passed")

def test_stage2_metrics_strength(min_acc=0.50):
    # guardrail: require at least 50% accuracy by default (tune for your dataset)
    assert stage2_metrics['accuracy'] >= min_acc, f"Stage-2 accuracy too low: {stage2_metrics['accuracy']:.3f}"
    print("✅ test_stage2_metrics_strength passed")

test_structure()
test_models_present()
test_stage2_metrics_strength(min_acc=0.50)  # adjust threshold as needed
